# M2: Ingestion Milestone Verification

This notebook verifies that the ingestion pipeline successfully ingested course materials into MongoDB.

**Requirements:**
- Print all URLs that have been ingested
- Show statistics about ingested content
- Display sample data

In [1]:
import sys
sys.path.append('/workspace')

from ingestion.mongo_helper import MongoHelper

## 1. Connect to MongoDB

In [3]:
mongo = MongoHelper()
print("Connected to MongoDB")

Connected to MongoDB


## 2. Document Counts

In [4]:
counts = mongo.count_documents()

print("=" * 70)
print("DOCUMENT COUNTS")
print("=" * 70)
print(f"Webpages: {counts['webpages']}")
print(f"Videos:   {counts['videos']}")
print(f"Slides:   {counts['slides']}")
print(f"TOTAL:    {sum(counts.values())}")

DOCUMENT COUNTS
Webpages: 92
Videos:   0
Slides:   0
TOTAL:    92


## 3. Sample Webpage Data

In [5]:
webpage = mongo.db.webpages.find_one()

if webpage:
    print("=" * 70)
    print("SAMPLE WEBPAGE")
    print("=" * 70)
    print(f"Title: {webpage['title']}")
    print(f"URL: {webpage['url']}")
    print(f"Word Count: {webpage['metadata'].get('word_count', 'N/A')}")
    print(f"\nContent Preview (first 500 chars):")
    print(webpage['content'][:500])
    print("...")
else:
    print("No webpages found in database")

SAMPLE WEBPAGE
Title: Introduction to Artificial Intelligence – Engineering AI Agents
URL: https://pantelis.github.io/courses/ai/
Word Count: 141

Content Preview (first 500 chars):
Introduction to Artificial Intelligence – Engineering AI Agents
Introduction to Artificial Intelligence
Foundations
2D Perception
Large Language Models
Logical Reasoning
Task Planning
Markov Decision Processes
Reinforcement Learning
Start (in-person)
Start (online)
What this course is all about
Artificial Intelligence (AI) addresses one of the ultimate puzzles humans are trying to solve: How is it possible for a brain, whether biological or electronic, to perceive, understand, predict and manipu
...


## 4. ALL INGESTED URLs 
This section lists all URLs that were ingested, as required by the M2 milestone.

In [6]:
print("=" * 70)
print("ALL INGESTED URLs")
print("=" * 70)

all_urls = mongo.get_all_urls()

for doc_type, url in all_urls:
    print(f"[{doc_type:8s}] {url}")

print(f"\nTotal URLs: {len(all_urls)}")

ALL INGESTED URLs
[webpage ] https://pantelis.github.io/courses/ai/
[webpage ] https://pantelis.github.io/book/foundations/index.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/learning-problem/index.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/regression/linear-regression/index.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/empirical-risk/index.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/optimization/sgd/index.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/optimization/maximum-likelihood/marginal_maximum_likelihood.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/optimization/maximum-likelihood/mle-gaussian-parameters.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/optimization/maximum-likelihood/conditional_maximum_likelihood.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/classification/classification-intro/index.html
[webpage ] https://pantelis.git

## 5. Content Statistics

In [7]:
print("=" * 70)
print("CONTENT STATISTICS")
print("=" * 70)

total_words = 0
word_counts = []

for doc in mongo.db.webpages.find():
    wc = doc['metadata'].get('word_count', 0)
    total_words += wc
    word_counts.append(wc)

if counts['webpages'] > 0:
    print(f"Total words ingested: {total_words:,}")
    print(f"Average words per page: {total_words // counts['webpages']:,}")
    print(f"Min words per page: {min(word_counts):,}")
    print(f"Max words per page: {max(word_counts):,}")

CONTENT STATISTICS
Total words ingested: 779,529
Average words per page: 8,473
Min words per page: 6
Max words per page: 509,866


## 6. Topic Distribution

Let's see what topics were covered in the ingested content.

In [8]:
from collections import Counter
import re

# Extract topics from URLs
topics = []
for doc_type, url in mongo.get_all_urls():
    # Extract meaningful parts from URL
    parts = url.split('/')
    for part in parts:
        if part and part not in ['https:', '', 'pantelis.github.io', 'aiml-common', 'lectures', 'index.html']:
            topics.append(part.replace('-', ' ').replace('_', ' '))

topic_counts = Counter(topics)

print("=" * 70)
print("TOP 20 TOPICS IN INGESTED CONTENT")
print("=" * 70)

for topic, count in topic_counts.most_common(20):
    print(f"{count:3d} - {topic}")

TOP 20 TOPICS IN INGESTED CONTENT
 17 - scene understanding
 17 - nlp
 12 - mdp
  9 - semantic segmentation
  9 - maskrcnn
  7 - book
  7 - reinforcement learning
  6 - object detection
  6 - logical reasoning
  6 - task planning
  5 - tf
  5 - nmt
  5 - planning
  4 - courses
  4 - ai
  4 - optimization
  4 - cnn
  4 - rnn
  4 - nlp introduction
  4 - language models


## 7. Sample Content from Different Topics

In [9]:
# Show samples from different topic areas
topics_to_sample = ['cnn', 'reinforcement-learning', 'nlp', 'mdp']

print("=" * 70)
print("SAMPLE CONTENT FROM DIFFERENT TOPICS")
print("=" * 70)

for topic in topics_to_sample:
    doc = mongo.db.webpages.find_one({'url': {'$regex': topic}})
    if doc:
        print(f"\n{'='*70}")
        print(f"Topic: {topic.upper()}")
        print(f"{'='*70}")
        print(f"Title: {doc['title']}")
        print(f"URL: {doc['url']}")
        print(f"\nFirst 300 characters:")
        print(doc['content'][:300])
        print("...")

SAMPLE CONTENT FROM DIFFERENT TOPICS

Topic: CNN
Title: index – Engineering AI Agents
URL: https://pantelis.github.io/aiml-common/lectures/cnn/cnn-example-architectures/index.html

First 300 characters:
index – Engineering AI Agents
CNN Example Architectures
This is a very high level view of practical structures of CNNs before the advent of more innovative architectures such as ResNets.
Toy CNN Network
convnet
The example CNN architecture above has the following layers:
INPUT [32x32x3] will hold th
...

Topic: REINFORCEMENT-LEARNING
Title: \epsilon-greedy Monte-Carlo (MC) Control – Engineering AI Agents
URL: https://pantelis.github.io/aiml-common/lectures/reinforcement-learning/model-free-control/greedy-monte-carlo/index.html

First 300 characters:
\epsilon-greedy Monte-Carlo (MC) Control – Engineering AI Agents
In this section we outline methods that can result in optimal policies when the MDP is
unknown
and we need to
learn
its underlying functions / models - also known as the
model 